## Inspect BatteryLife Dataset

This notebook inspects the raw `batterylife` sodium-ion dataset stored in `data/raw/batterylife/set/naion` (as for now only naion is present). The goal is to document the file structure, check basic consistency across batteries and cycles, inspect sampling anomalies, and review cycle-level signals before building downstream preprocessing.

### General Data Structure

This section lists the raw `.pkl` files in the `naion` subset and loads one example file to inspect its nested schema. The preview is intentionally truncated so repeated list structures stay readable while preserving the names of the recorded fields.

In [ ]:
import os

datapath = "set/naion"

# list files in the directory
files = os.listdir(datapath)
len(files), files[:3]

In [ ]:
import pickle
import json

fiepath = os.path.join(datapath, files[0])

with open(fiepath, "rb") as f:
    data = pickle.load(f)


def _is_leaf_list(lst):
    return all(not isinstance(item, (list, dict)) for item in lst)


def _shape_signature(obj):
    if isinstance(obj, dict):
        return {
            k: _shape_signature(v)
            for k, v in sorted(obj.items(), key=lambda kv: str(kv[0]))
        }
    if isinstance(obj, list):
        if not obj:
            return ["empty_list"]
        return ["list", _shape_signature(obj[0])]
    return type(obj).__name__


def _is_homogeneous_struct_list(lst):
    if len(lst) <= 1:
        return False
    if not all(isinstance(item, (dict, list)) for item in lst):
        return False

    first_shape = _shape_signature(lst[0])
    return all(_shape_signature(item) == first_shape for item in lst[1:])


def truncate_preview(obj, keep_leaf=3):
    if isinstance(obj, dict):
        return {k: truncate_preview(v, keep_leaf=keep_leaf) for k, v in obj.items()}

    if isinstance(obj, list):
        if _is_homogeneous_struct_list(obj):
            more = len(obj) - 1
            return [
                truncate_preview(obj[0], keep_leaf=keep_leaf),
                f"... ({more} more like this)",
            ]

        if _is_leaf_list(obj) and len(obj) > keep_leaf:
            more = len(obj) - keep_leaf
            return obj[:keep_leaf] + [f"... ({more} more items)"]

        return [truncate_preview(item, keep_leaf=keep_leaf) for item in obj]

    return obj


# inspect its schema
preview = truncate_preview(data, keep_leaf=3)
print(json.dumps(preview, indent=2, ensure_ascii=False, default=str))

### Consistency Check

This section scans all battery files to summarize three structural properties: sample-to-sample timestep, cycle duration, and number of cycles per battery. The printed statistics and histograms are used to confirm that the recording cadence is mostly stable and to identify cycles that may require special handling during preprocessing.

In [ ]:
import math
import pickle
from statistics import median


timestep_sizes_s = []
cycle_durations_s = []
cycles_count_per_battery = []


for filename in files:
    filepath = os.path.join(datapath, filename)
    with open(filepath, "rb") as f:
        battery_data = pickle.load(f)

    cycles = battery_data.get("cycle_data", [])
    cycles_count_per_battery.append(len(cycles))

    for cycle in cycles:
        time_values = cycle.get("time_in_s") or []
        if len(time_values) < 2:
            continue

        cycle_durations_s.append(float(time_values[-1]) - float(time_values[0]))

        for t_prev, t_next in zip(time_values[:-1], time_values[1:]):
            dt = float(t_next) - float(t_prev)
            if math.isfinite(dt) and dt >= 0:
                timestep_sizes_s.append(dt)


print(f"Batteries processed: {len(files)}")
print(f"Cycles processed: {sum(cycles_count_per_battery)}")
print(f"Collected timestep deltas: {len(timestep_sizes_s)}")
print(f"Collected cycle durations: {len(cycle_durations_s)}")

if timestep_sizes_s:
    print(
        "Timestep size [s] stats:",
        f"min={min(timestep_sizes_s):.4f}, median={median(timestep_sizes_s):.4f}, max={max(timestep_sizes_s):.4f}",
    )
if cycle_durations_s:
    print(
        "Cycle duration [s] stats:",
        f"min={min(cycle_durations_s):.2f}, median={median(cycle_durations_s):.2f}, max={max(cycle_durations_s):.2f}",
    )
if cycles_count_per_battery:
    print(
        "Cycles per battery stats:",
        f"min={min(cycles_count_per_battery)}, median={median(cycles_count_per_battery):.1f}, max={max(cycles_count_per_battery)}",
    )

In [ ]:
def cycle_discharge_relation_from_current(cycle):
    time_values = cycle.get("time_in_s") or []
    current_values = cycle.get("current_in_A") or []

    if len(time_values) < 2 or len(current_values) < 2:
        return None
    if len(time_values) != len(current_values):
        return None

    time2charge = 0.0
    time2discharge = 0.0

    for time_prev, time_next, current_value in zip(
        time_values[:-1],
        time_values[1:],
        current_values[:-1],
    ):
        if current_value is None:
            continue

        dt = float(time_next) - float(time_prev)
        current_value = float(current_value)

        if not math.isfinite(dt) or dt <= 0:
            continue
        if not math.isfinite(current_value) or current_value == 0:
            continue

        if current_value > 0:
            time2charge += dt
        else:
            time2discharge += dt

    total_active_time = time2charge + time2discharge
    if total_active_time <= 0:
        return None

    return time2discharge / total_active_time


discharge_relations = []

for filename in files:
    filepath = os.path.join(datapath, filename)
    with open(filepath, "rb") as f:
        battery_data = pickle.load(f)

    for cycle in battery_data.get("cycle_data", []):
        discharge_relation = cycle_discharge_relation_from_current(cycle)
        if discharge_relation is not None:
            discharge_relations.append(discharge_relation)


def percentile(sorted_values, q):
    if not sorted_values:
        return None
    if len(sorted_values) == 1:
        return sorted_values[0]
    position = q * (len(sorted_values) - 1)
    lower_index = int(position)
    upper_index = min(lower_index + 1, len(sorted_values) - 1)
    weight = position - lower_index
    return (
        sorted_values[lower_index] * (1.0 - weight)
        + sorted_values[upper_index] * weight
    )


print(f"Collected discharge_relation values: {len(discharge_relations)}")

if discharge_relations:
    sorted_relations = sorted(discharge_relations)
    print(
        "discharge_relation stats:",
        f"min={sorted_relations[0]:.6f}, "
        f"p05={percentile(sorted_relations, 0.05):.6f}, "
        f"p25={percentile(sorted_relations, 0.25):.6f}, "
        f"median={median(sorted_relations):.6f}, "
        f"p75={percentile(sorted_relations, 0.75):.6f}, "
        f"p95={percentile(sorted_relations, 0.95):.6f}, "
        f"max={sorted_relations[-1]:.6f}",
    )

    print("discharge_relation histogram (all cycles):")
    bin_edges = [i / 10 for i in range(11)]
    for left_edge, right_edge in zip(bin_edges[:-1], bin_edges[1:]):
        if right_edge < 1.0:
            count = sum(
                left_edge <= value < right_edge for value in discharge_relations
            )
            label = f"[{left_edge:.1f}, {right_edge:.1f})"
        else:
            count = sum(
                left_edge <= value <= right_edge for value in discharge_relations
            )
            label = f"[{left_edge:.1f}, {right_edge:.1f}]"
        print(f"  {label}: {count}")

In [ ]:
import plotly.graph_objects as go

fig_dt = go.Figure(
    data=[go.Histogram(x=timestep_sizes_s, nbinsx=120, name="dt")],
)
fig_dt.update_layout(
    title="Timestep Size Distribution Across All Cycles and Batteries",
    xaxis_title="Timestep size (s)",
    yaxis_title="Count",
    template="plotly_white",
)
fig_dt.show()

In [ ]:
import math
import pickle
from collections import defaultdict

lower_dt = 0.99
upper_dt = 1.01

anomalies = []


def safe_at(values, idx):
    if idx < len(values):
        value = values[idx]
        if value is None:
            return None
        return float(value)
    return None


for battery_index, filename in enumerate(files):
    filepath = os.path.join(datapath, filename)
    with open(filepath, "rb") as f:
        battery_data = pickle.load(f)

    for cycle_index, cycle in enumerate(battery_data.get("cycle_data", [])):
        time_values = cycle.get("time_in_s") or []
        voltage_values = cycle.get("voltage_in_V") or []
        current_values = cycle.get("current_in_A") or []

        # check fatal anomalies
        assert len(time_values) == len(voltage_values) == len(current_values)
        assert (
            len(time_values) >= 2
        ), f"Cycle {cycle_index} in {filename} has less than 2 time points"
        assert cycle_index + 1 == cycle.get(
            "cycle_number"
        ), f"Cycle {cycle_index} in {filename} has inconsistent cycle number"

        for sample_index, (t_prev, t_next) in enumerate(
            zip(time_values[:-1], time_values[1:])
        ):
            dt = float(t_next) - float(t_prev)
            if not math.isfinite(dt) or dt < 0:
                continue

            if dt < lower_dt or dt > upper_dt:
                anomalies.append(
                    {
                        "dt": dt,
                        "battery_index": battery_index,
                        "battery_file": filename,
                        "cycle_index": cycle_index,
                        "sample_index": sample_index,
                        "voltage_in_V": safe_at(voltage_values, sample_index),
                        "current_in_A": safe_at(current_values, sample_index),
                    }
                )


def compress_rows(rows, max_gap=2):
    rows = sorted(rows, key=lambda row: row["sample_index"])
    if not rows:
        return []

    groups = [[rows[0]]]

    for row in rows[1:]:
        prev = groups[-1][-1]
        if row["sample_index"] - prev["sample_index"] <= max_gap:
            groups[-1].append(row)
        else:
            groups.append([row])

    return groups


def format_index_span(rows):
    if len(rows) == 1:
        return str(rows[0]["sample_index"])
    return f"{rows[0]['sample_index']}-{rows[-1]['sample_index']}({len(rows)})"


def format_series(label, rows, key, digits=6):
    values = []
    for row in rows:
        value = row[key]
        if value is None or not math.isfinite(value):
            values.append("None")
        else:
            values.append(f"{value:.{digits}f}")

    if len(values) == 1:
        return f"{label}={values[0]}"
    return f"{label}=[{', '.join(values)}]"


def print_compressed_anomalies(
    anomalies,
    dt_digits=1,
    value_digits=6,
    max_gap=2,
):
    grouped = defaultdict(lambda: defaultdict(list))

    for row in anomalies:
        dt_key = round(float(row["dt"]), dt_digits)
        grouped[dt_key][(row["battery_file"], row["cycle_index"])].append(row)

    for dt_value in sorted(grouped):
        print(f"dt: {dt_value:.{dt_digits}f}")
        for (battery_file, cycle_index), rows in sorted(grouped[dt_value].items()):
            for block in compress_rows(rows, max_gap=max_gap):
                span = format_index_span(block)
                voltage_text = format_series(
                    "V", block, "voltage_in_V", digits=value_digits
                )
                current_text = format_series(
                    "I", block, "current_in_A", digits=value_digits
                )
                print(
                    f"{battery_file} {cycle_index} {span} {voltage_text} {current_text}"
                )
        print()


print_compressed_anomalies(anomalies, dt_digits=1, value_digits=6, max_gap=2)

In [ ]:
import os
import pickle

target_file = "NA-ion_270040-1-1-64.pkl"
# target_file = "NA-ion_2850-30_20250117105706_DefaultGroup_45_2.pkl"
filepath = os.path.join(datapath, target_file)

with open(filepath, "rb") as f:
    battery_data = pickle.load(f)

print("cycle_number\tstart_time_s\tend_time_s\tduration_s")

for cycle_index, cycle in enumerate(battery_data.get("cycle_data", [])):
    time_values = cycle.get("time_in_s") or []
    if not time_values:
        print(f"{cycle.get('cycle_number')}\t\tNone\t\tNone\t\tNone")
        continue

    start_time = float(time_values[0])
    end_time = float(time_values[-1])
    duration = end_time - start_time

    if cycle_index == 5:
        print("...")

    if cycle_index >= 5:
        continue  # only print the first few cycles for brevity
    print(
        f"{cycle.get('cycle_number')}\t\t"
        f"{start_time:.6f}\t{end_time:.6f}\t{duration:.6f}"
    )

> Note: In the inspected example, the start time of each cycle appears to match the end time of the previous cycle, so cycle boundaries look temporally continuous.

In [ ]:
# Inspect the sample range around the detected anomaly closer

import os
import pickle

import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_battery_cycle_sample_range(
    battery_file,
    cycle_index,
    sample_start,
    sample_end,
):
    filepath = os.path.join(datapath, battery_file)

    with open(filepath, "rb") as f:
        battery_data = pickle.load(f)

    cycles = battery_data.get("cycle_data", [])
    assert 0 <= cycle_index < len(cycles), (
        f"cycle_index={cycle_index} out of range, " f"available: 0..{len(cycles) - 1}"
    )

    cycle = cycles[cycle_index]
    time_values = cycle.get("time_in_s") or []
    voltage_values = cycle.get("voltage_in_V") or []
    current_values = cycle.get("current_in_A") or []

    assert (
        len(time_values) == len(voltage_values) == len(current_values)
    ), "time/voltage/current lengths do not match"

    n = len(time_values)
    assert n > 0, "selected cycle has no samples"

    sample_start = max(0, sample_start)
    sample_end = min(sample_end, n - 1)
    assert sample_start <= sample_end, "sample_start must be <= sample_end"

    sample_idx = list(range(sample_start, sample_end + 1))
    time_slice = time_values[sample_start : sample_end + 1]
    voltage_slice = voltage_values[sample_start : sample_end + 1]
    current_slice = current_values[sample_start : sample_end + 1]

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=("time_in_s", "voltage_in_V", "current_in_A"),
    )

    fig.add_trace(
        go.Scatter(
            x=sample_idx,
            y=time_slice,
            mode="lines+markers",
            name="time_in_s",
            line={"color": "#4c78a8"},
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=sample_idx,
            y=voltage_slice,
            mode="lines+markers",
            name="voltage_in_V",
            line={"color": "#f58518"},
        ),
        row=2,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=sample_idx,
            y=current_slice,
            mode="lines+markers",
            name="current_in_A",
            line={"color": "#e45756"},
        ),
        row=3,
        col=1,
    )

    fig.update_xaxes(title_text="sample index", row=3, col=1)
    fig.update_yaxes(title_text="time (s)", row=1, col=1)
    fig.update_yaxes(title_text="voltage (V)", row=2, col=1)
    fig.update_yaxes(title_text="current (A)", row=3, col=1)

    fig.update_layout(
        height=900,
        template="plotly_white",
        title=(
            f"{battery_file}, cycle_index={cycle_index}, "
            f"cycle_number={cycle.get('cycle_number')}, "
            f"samples={sample_start}:{sample_end}"
        ),
        showlegend=False,
    )

    fig.show()


# plot_battery_cycle_sample_range(
#     battery_file="NA-ion_270040-1-1-64.pkl",
#     cycle_index=0,
#     sample_start=1350,
#     sample_end=1405,
# )
# plot_battery_cycle_sample_range(
#     battery_file="NA-ion_270040-1-1-64.pkl",
#     cycle_index=0,
#     sample_start=0,
#     sample_end=10,
# )
plot_battery_cycle_sample_range(
    # dt=3.0
    battery_file="NA-ion_2850-30_20250117105706_DefaultGroup_45_2.pkl",
    cycle_index=0,
    sample_start=1,
    sample_end=300,
)

> Note: The dataset contains small sampling-rate irregularities, including occasional `dt = 0` values and short gaps larger than one second. For uniform resampling, these local issues should be manageable with interpolation, and duplicate timestamps can be merged by averaging the corresponding values.

In [ ]:
fig_cycle_time = go.Figure(
    data=[go.Histogram(x=cycle_durations_s, nbinsx=120, name="cycle_time")],
)
fig_cycle_time.update_layout(
    title="Cycle Time Distribution Across All Cycles",
    xaxis_title="Cycle duration (s)",
    yaxis_title="Count",
    template="plotly_white",
)
fig_cycle_time.show()

In [ ]:
fig_cycle_count = go.Figure(
    data=[
        go.Histogram(
            x=cycles_count_per_battery,
            nbinsx=min(50, max(10, len(set(cycles_count_per_battery)))),
            name="cycles_per_battery",
        )
    ],
)
fig_cycle_count.update_layout(
    title="Cycles Count Distribution Across Batteries",
    xaxis_title="Number of cycles per battery",
    yaxis_title="Count",
    template="plotly_white",
)
fig_cycle_count.show()

### Detailed Signal Inspection

This section provides interactive browsers for cycle-level capacity, voltage, and current traces. It is intended for inspecting representative charge-discharge profiles, comparing batteries across cycles, and spotting recording quirks that are not obvious from aggregate statistics alone.

In [ ]:
import pickle

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import clear_output, display


battery_cache = {}
cycle_cache = {}


def load_battery(file_index):
    if file_index not in battery_cache:
        filepath = os.path.join(datapath, files[file_index])
        with open(filepath, "rb") as f:
            battery_cache[file_index] = pickle.load(f)
    return battery_cache[file_index]


def get_valid_cycles(file_index):
    if file_index not in cycle_cache:
        battery_data = load_battery(file_index)
        valid_cycles = []
        for cycle in battery_data.get("cycle_data", []):
            if any(cycle.get(metric_key) for metric_key in cycle.keys()):
                valid_cycles.append(cycle)
        if not valid_cycles:
            raise ValueError(f"No cycle data found for battery {file_index}.")
        cycle_cache[file_index] = valid_cycles
    return cycle_cache[file_index]


def relative_position(index, count):
    if count <= 1:
        return 0.0
    return index / (count - 1)


def index_from_relative(pos, count):
    if count <= 1:
        return 0
    return round(pos * (count - 1))


def make_metric_figure(
    battery_index,
    cycle_index,
    metric_keys,
    metric_labels=None,
    color_map=None,
    yaxis_title="Value",
    title_label=None,
    template="plotly_white",
):
    valid_cycles = get_valid_cycles(battery_index)
    cycle = valid_cycles[cycle_index]
    cycle_number = cycle.get("cycle_number")
    time_values = cycle.get("time_in_s") or []

    metric_labels = metric_labels or {}
    color_map = color_map or {}
    palette = ["#1b9e77", "#d95f02", "#7570b3", "#e7298a", "#66a61e", "#e6ab02"]

    fig = go.Figure()
    uses_time_axis = False

    for i, metric_key in enumerate(metric_keys):
        values = cycle.get(metric_key) or []
        if not values:
            continue

        metric_name = metric_labels.get(metric_key, metric_key)
        color = color_map.get(metric_key, palette[i % len(palette)])

        if time_values:
            n = min(len(time_values), len(values))
            if n == 0:
                continue
            x_values = time_values[:n]
            y_values = values[:n]
            uses_time_axis = True
        else:
            x_values = list(range(len(values)))
            y_values = values

        fig.add_trace(
            go.Scatter(
                x=x_values,
                y=y_values,
                mode="lines",
                name=metric_name,
                line={"color": color},
            )
        )

    if not fig.data:
        fig.add_annotation(
            text="No values for selected metrics in this cycle",
            xref="paper",
            yref="paper",
            x=0.5,
            y=0.5,
            showarrow=False,
        )

    metrics_text = title_label or ", ".join(
        metric_labels.get(k, k) for k in metric_keys
    )
    fig.update_layout(
        title=(
            f"Battery {battery_index + 1}/{len(files)} ({files[battery_index]}), "
            f"cycle {cycle_number}: {metrics_text}"
        ),
        xaxis_title="Time (s)" if uses_time_axis else "Sample index",
        yaxis_title=yaxis_title,
        template=template,
    )
    return fig


def display_metric_browser(
    metric_keys,
    metric_labels=None,
    color_map=None,
    yaxis_title="Value",
    title_label=None,
    template="plotly_white",
):
    metric_labels = metric_labels or {}
    color_map = color_map or {}

    battery_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(files) - 1,
        step=1,
        description="Battery",
        continuous_update=False,
    )
    cycle_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=0,
        step=1,
        description="Cycle idx",
        continuous_update=False,
    )
    battery_text = widgets.HTML()
    cycle_text = widgets.HTML()
    plot_output = widgets.Output()

    def set_cycle_for_battery(battery_index, keep_relative=None):
        valid_cycles = get_valid_cycles(battery_index)
        new_count = len(valid_cycles)
        if keep_relative is None:
            new_idx = min(cycle_slider.value, new_count - 1)
        else:
            new_idx = index_from_relative(keep_relative, new_count)

        new_idx = min(max(new_idx, 0), new_count - 1)
        with cycle_slider.hold_trait_notifications():
            cycle_slider.max = new_count - 1
            cycle_slider.value = new_idx

    def update_labels():
        b_idx = battery_slider.value
        valid_cycles = get_valid_cycles(b_idx)
        c_idx = cycle_slider.value
        cycle_number = valid_cycles[c_idx].get("cycle_number")
        battery_text.value = f"<b>Battery idx:</b> {b_idx + 1}/{len(files)}"
        cycle_text.value = (
            f"<b>Cycle idx:</b> {c_idx + 1}/{len(valid_cycles)}"
            f" &nbsp; <b>Cycle number:</b> {cycle_number}"
        )

    def render_plot(*_):
        update_labels()
        with plot_output:
            clear_output(wait=True)
            fig = make_metric_figure(
                battery_index=battery_slider.value,
                cycle_index=cycle_slider.value,
                metric_keys=metric_keys,
                metric_labels=metric_labels,
                color_map=color_map,
                yaxis_title=yaxis_title,
                title_label=title_label,
                template=template,
            )
            fig.show()

    def on_battery_change(change):
        old_battery = change.get("old", change["new"])
        old_count = len(get_valid_cycles(old_battery))
        old_rel = relative_position(cycle_slider.value, old_count)
        set_cycle_for_battery(change["new"], keep_relative=old_rel)
        render_plot()

    def on_cycle_change(change):
        if change.get("name") == "value":
            render_plot()

    set_cycle_for_battery(battery_slider.value)
    battery_slider.observe(on_battery_change, names="value")
    cycle_slider.observe(on_cycle_change, names="value")
    render_plot()

    display(
        widgets.VBox(
            [
                battery_slider,
                battery_text,
                cycle_slider,
                cycle_text,
                plot_output,
            ]
        )
    )

In [ ]:
display_metric_browser(
    metric_keys=["discharge_capacity_in_Ah", "charge_capacity_in_Ah"],
    metric_labels={
        "discharge_capacity_in_Ah": "discharge_capacity_in_Ah",
        "charge_capacity_in_Ah": "charge_capacity_in_Ah",
    },
    color_map={
        "discharge_capacity_in_Ah": "#1b9e77",
        "charge_capacity_in_Ah": "#d95f02",
    },
    yaxis_title="Capacity (Ah)",
    title_label="charge/discharge capacity",
)

> Note: Files with `DefaultGroup` in the name use a different convention for recorded charge and discharge capacity. They are planned to be excluded from capacity-based analysis because the same phase information can be inferred from the current profile.

In [ ]:
display_metric_browser(
    metric_keys=["voltage_in_V", "current_in_A"],
    metric_labels={
        "voltage_in_V": "voltage_in_V",
        "current_in_A": "current_in_A",
    },
    color_map={
        "voltage_in_V": "#377eb8",
        "current_in_A": "#e41a1c",
    },
    yaxis_title="Value (mixed units)",
    title_label="voltage/current profile",
)

### Population-Level Capacity Picture

This section compares each battery's discharge-capacity trajectory against the population mean and an asymmetric spread band over cycle number. The interactive view helps identify typical degradation behavior, dispersion across cells, and batteries that diverge from the broader trend.

In [ ]:
from collections import defaultdict
import math

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
from IPython.display import clear_output, display


def finite_float_list(values):
    out = []
    for value in values or []:
        if value is None:
            continue
        value = float(value)
        if math.isfinite(value):
            out.append(value)
    return out


def cycle_discharge_capacity(cycle, reduction="max"):
    values = finite_float_list(cycle.get("discharge_capacity_in_Ah"))
    if not values:
        return None

    if reduction == "last":
        return values[-1]
    if reduction == "max":
        return max(values)

    raise ValueError(f"Unsupported reduction: {reduction}")


def build_battery_discharge_series(file_index, reduction="max"):
    cycles = get_valid_cycles(file_index)
    series = []

    for cycle_index, cycle in enumerate(cycles):
        cycle_number = cycle.get("cycle_number", cycle_index + 1)
        capacity = cycle_discharge_capacity(cycle, reduction=reduction)
        if capacity is None:
            continue

        series.append(
            {
                "cycle_number": int(cycle_number),
                "capacity": float(capacity),
            }
        )

    return series


def build_population_discharge_stats(all_series, band_scale=1.0):
    values_by_cycle = defaultdict(list)

    for series in all_series:
        for row in series:
            values_by_cycle[row["cycle_number"]].append(row["capacity"])

    stats = []
    for cycle_number in sorted(values_by_cycle):
        values = np.asarray(values_by_cycle[cycle_number], dtype=float)
        mean_value = float(values.mean())

        lower_values = values[values <= mean_value]
        upper_values = values[values >= mean_value]

        lower_std = (
            float(np.sqrt(np.mean((mean_value - lower_values) ** 2)))
            if lower_values.size
            else 0.0
        )
        upper_std = (
            float(np.sqrt(np.mean((upper_values - mean_value) ** 2)))
            if upper_values.size
            else 0.0
        )

        stats.append(
            {
                "cycle_number": cycle_number,
                "mean": mean_value,
                "lower_band": mean_value - band_scale * lower_std,
                "upper_band": mean_value + band_scale * upper_std,
                "count": int(values.size),
            }
        )

    return stats


def display_discharge_capacity_browser(reduction="max", band_scale=1.0):
    all_series = [
        build_battery_discharge_series(file_index, reduction=reduction)
        for file_index in range(len(files))
    ]
    population_stats = build_population_discharge_stats(
        all_series, band_scale=band_scale
    )

    battery_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(files) - 1,
        step=1,
        description="Battery",
        continuous_update=False,
    )
    battery_text = widgets.HTML()
    plot_output = widgets.Output()

    def render_plot(*_):
        file_index = battery_slider.value
        battery_text.value = (
            f"<b>Battery idx:</b> {file_index + 1}/{len(files)}"
            f" &nbsp; <b>File:</b> {files[file_index]}"
        )

        battery_series = all_series[file_index]

        fig = go.Figure()

        if population_stats:
            x_pop = [row["cycle_number"] for row in population_stats]
            y_lower = [row["lower_band"] for row in population_stats]
            y_upper = [row["upper_band"] for row in population_stats]
            y_mean = [row["mean"] for row in population_stats]
            counts = [row["count"] for row in population_stats]

            fig.add_trace(
                go.Scatter(
                    x=x_pop,
                    y=y_upper,
                    mode="lines",
                    line={"width": 0},
                    hoverinfo="skip",
                    showlegend=False,
                    name="upper band",
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=x_pop,
                    y=y_lower,
                    mode="lines",
                    line={"width": 0},
                    fill="tonexty",
                    fillcolor="rgba(55, 126, 184, 0.18)",
                    hoverinfo="skip",
                    name=f"population asymmetric band ({band_scale:.1f}x)",
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=x_pop,
                    y=y_mean,
                    mode="lines",
                    name="population mean",
                    line={"color": "#377eb8", "width": 2},
                    customdata=np.asarray(counts).reshape(-1, 1),
                    hovertemplate=(
                        "cycle=%{x}<br>"
                        "mean capacity=%{y:.6f}<br>"
                        "battery count=%{customdata[0]}<extra></extra>"
                    ),
                )
            )

        if battery_series:
            x_battery = [row["cycle_number"] for row in battery_series]
            y_battery = [row["capacity"] for row in battery_series]

            fig.add_trace(
                go.Scatter(
                    x=x_battery,
                    y=y_battery,
                    mode="lines+markers",
                    name=files[file_index],
                    line={"color": "#e41a1c", "width": 3},
                    marker={"size": 5},
                    hovertemplate=(
                        "cycle=%{x}<br>" "battery capacity=%{y:.6f}<extra></extra>"
                    ),
                )
            )

        fig.update_layout(
            title=("Discharge Capacity vs Cycle Number" f" ({files[file_index]})"),
            xaxis_title="Cycle number",
            yaxis_title="Discharge capacity (Ah)",
            template="plotly_white",
            height=600,
        )

        with plot_output:
            clear_output(wait=True)
            fig.show()

    battery_slider.observe(render_plot, names="value")
    render_plot()

    display(
        widgets.VBox(
            [
                battery_slider,
                battery_text,
                plot_output,
            ]
        )
    )


display_discharge_capacity_browser(reduction="max", band_scale=1.0)